##In this notebook, we simulate rainfall-runoff accross 222 CAMELS-AUS catchments using the GRHyMoLAP model.

# IMPORT LIBRARIES

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from matplotlib.dates import DateFormatter
from scipy.optimize import minimize # USE IN THE MODEL CALIBRATION

from google.colab import files
import zipfile
import os

In [ ]:
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##CAMELS-DATA from my Google Drive.

In [ ]:
# ==============================
# Paths to ZIP files
# ==============================
hydro_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/05_hydrometeorology.zip"
streamflow_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/03_streamflow.zip"

# Data extraction directories
hydro_dir = "/content/05_hydro"
streamflow_dir = "/content/03_streamflow"

# ==============================
# Function to extract ZIP files
# ==============================
def extract_zip(zip_path, extract_to):
    if not os.path.exists(extract_to):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✅ ZIP extracted in {extract_to}")
    else:
        print(f"✅ Directory {extract_to} already exists")

def find_csv(base_dir, csv_name):
    # Recursive search for the CSV file
    for root, dirs, files in os.walk(base_dir):
        if csv_name in files:
            return os.path.join(root, csv_name)
    raise FileNotFoundError(f"{csv_name} not found in {base_dir}")

# ==============================
# Extraction
# ==============================
extract_zip(hydro_zip, hydro_dir)
extract_zip(streamflow_zip, streamflow_dir)

# ==============================
# Load the 222 basins data
# ==============================
file_path = '/content/drive/MyDrive/Colab Notebooks/Dimension/id_name_metadata.csv'
basin222 = pd.read_csv(file_path)
station_ids_v1 = basin222['station_id'].astype(str).str.strip().unique()
print(f"✅ {len(station_ids_v1)} official basins loaded")

# ==============================
# 1️⃣ Precipitation (SILO)
# ==============================
precip_file = find_csv(hydro_dir, "precipitation_SILO.csv")
precip = pd.read_csv(precip_file, index_col=0, parse_dates=True)
precip.columns = precip.columns.str.strip()
precip.replace(-99.99, np.nan, inplace=True)
print("✅ SILO precipitation:", precip.shape)

# ==============================
# 2️⃣ Evapotranspiration (ET SILO)
# ==============================
et_file = find_csv(hydro_dir, "et_morton_actual_SILO.csv")
et = pd.read_csv(et_file, index_col=0, parse_dates=True)
et.columns = et.columns.str.strip()
et.replace(-99.99, np.nan, inplace=True)
print("✅ SILO ET:", et.shape)

# ==============================
# 3️⃣ Streamflow
# ==============================
streamflow_file = find_csv(streamflow_dir, "streamflow_mmd.csv")
Q = pd.read_csv(streamflow_file, index_col=0, parse_dates=True)
Q.columns = Q.columns.str.strip()
Q.replace(-99.99, np.nan, inplace=True)
print("✅ Streamflow:", Q.shape)

# ==============================
# 4️⃣ Identify common stations
# ==============================
stations_precip = set(precip.columns)
stations_et = set(et.columns)
stations_Q = set(Q.columns)

common_stations = [
    s for s in station_ids_v1
    if s in stations_precip and s in stations_et and s in stations_Q
]

print(f"✅ Official common stations: {len(common_stations)}")

# ==============================
# 5️⃣ Subset common stations
# ==============================
precip = precip[common_stations]
et = et[common_stations]
Q = Q[common_stations]

# ==============================
# 6️⃣ Final verification
# ==============================
print("Precipitation:", precip.shape)
print("ET:", et.shape)
print("Streamflow:", Q.shape)
print("Stations (first 10):", common_stations[:10], "...")


✅ Directory /content/05_hydro already exists
✅ Directory /content/03_streamflow already exists
✅ 222 official basins loaded
✅ SILO precipitation: (43464, 224)
✅ SILO ET: (43464, 224)
✅ Streamflow: (23376, 224)
✅ Official common stations: 222
Precipitation: (43464, 222)
ET: (43464, 222)
Streamflow: (23376, 222)
Stations (first 10): ['912101A', '912105A', '915011A', '917107A', '919003A', '919201A', '919309A', '922101B', '925001A', '926002A'] ...


In [ ]:
# Verification of the periods
print("Precipitation :", precip.index.min(), "→", precip.index.max())
print("ET            :", et.index.min(), "→", et.index.max())
print("Streamflow    :", Q.index.min(), "→", Q.index.max())


Precipitation : 1900-01-01 00:00:00 → 2018-01-01 00:00:00
ET            : 1900-01-01 00:00:00 → 2018-01-01 00:00:00
Streamflow    : 1951-01-01 00:00:00 → 2014-01-01 00:00:00


In [ ]:
# ==============================
# 0️⃣ Reduce all series to the period 1 January 1980 → 31 December 2014
# ==============================
start_date = "1980-01-01"
end_date   = "2014-12-31"

precip = precip.loc[start_date:end_date]
et     = et.loc[start_date:end_date]
Q      = Q.loc[start_date:end_date]

# Verification
print("Common period verification:")
print("Precipitation :", precip.index.min(), "→", precip.index.max())
print("ET            :", et.index.min(), "→", et.index.max())
print("Streamflow    :", Q.index.min(), "→", Q.index.max())

Common period verification:
Precipitation : 1980-01-01 00:00:00 → 2014-01-01 00:00:00
ET            : 1980-01-01 00:00:00 → 2014-01-01 00:00:00
Streamflow    : 1980-01-01 00:00:00 → 2014-01-01 00:00:00


GRHyMoLAP

In [ ]:
# ============================================
# General parameters
# ============================================
start_date = "1980-01-01"
end_date = "2014-12-31"
b1_ratio = 0.7
max_missing_ratio = 1

stations = common_stations  # The 222 official basins
results = {}

# ============================================
# Main loop over each basin
# ============================================
i = 0
for station_id in stations:
    i = i + 1
    print(f"\n=== Station {station_id} ===, Number = ", i)

    # Extract time series
    Q_obs = Q[station_id].loc[start_date:end_date].to_numpy(float)
    P = precip[station_id].loc[start_date:end_date].to_numpy(float)
    PET = et[station_id].loc[start_date:end_date].to_numpy(float)

    Pn = np.maximum(0, P - PET)  # Net precipitation
    En = np.maximum(0, PET - P)  # Excess evapotranspiration

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        print(f"⚠️ Station skipped (no valid data).")
        continue

    missing_count = np.sum(np.isnan(Q_obs))
    missing_ratio = missing_count / N
    if missing_ratio > max_missing_ratio:
        print(f"⚠️ Too many missing values ({missing_ratio*100:.1f}%)")
        continue

    b1 = int(N * b1_ratio)
    Q0 = Q_obs[0]

    # ============================================
    # Percolation
    # ============================================
    def Percolation(Pn, En, X1):
        n = len(Pn)
        S = np.zeros(n)
        S[0] = X1/2
        Perc = np.zeros(n)
        ratio = (4.0 / 9.0) * (S[0] / X1)
        Perc[0] = S[0] * (1 - (1 + ratio**4) ** (-0.25))
        for i in range(1, n):
            temp = (S[i-1] / X1) ** 2
            frac = Pn[i] / X1
            Ps = X1 * (1 - temp) * np.tanh(frac) / (1 + (S[i-1] / X1) * np.tanh(frac))
            frac = En[i] / X1
            Es = S[i-1] * (2 - S[i-1]/X1) * np.tanh(frac) / (1 + (1 - S[i-1]/X1) * np.tanh(frac))
            S[i] = S[i-1] + Ps - Es
            ratio = (4.0 / 9.0) * (S[i] / X1)
            Perc[i] = S[i] * (1 - (1 + ratio**4) ** (-0.25))
            S[i] = S[i] - Perc[i]
        return Perc

    # ============================================
    # GRHyMoLAP model
    # ============================================
    def GRHyMoLAP_Model(params, Q0, Pn, En):
        MU, LAMBDA, X1, gamma = params
        N = len(Pn)
        Q = np.zeros(N)
        Q[0] = Q0
        Perc = Percolation(Pn, En, X1)
        for t in range(N-1):
            Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
        return Q

    # ============================================
    # NSE with NaN handling
    # ============================================
    def NSE(obs, sim):
        df = pd.DataFrame({"obs": obs, "sim": sim}).dropna()
        if df.empty or df["obs"].var() == 0:
            return np.nan
        return 1 - np.sum((df["sim"] - df["obs"])**2) / np.sum((df["obs"] - df["obs"].mean())**2)

    # ============================================
    # NNSE (Normalized NSE)
    # ============================================
    def NNSE(nse):
        if np.isnan(nse):
            return np.nan
        return 1.0 / (2.0 - nse)

    # ============================================
    # Objective function
    # ============================================
    def objective(params, Q0, Pn_train, En_train, Q_obs_train):
        Q_sim = GRHyMoLAP_Model(params, Q0, Pn_train, En_train)
        nse = NSE(Q_obs_train, Q_sim)
        return 1 - nse if np.isfinite(nse) else 1e9

    # ============================================
    # MULTI-START optimization
    # ============================================
    initial_guesses = [
        [1.0, 8, 150, 0.1],
        [0.6, 2, 120, 1],
        [1.4, 15, 200, 0.5]
    ]

    best_res = None
    best_val = float("inf")

    for guess in initial_guesses:
        res = minimize(
            objective, guess,
            args=(Q0, Pn[:b1], En[:b1], Q_obs[:b1]),
            method="Nelder-Mead",
            options={'maxiter': 2500, 'disp': False}
        )
        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    MU, LAMBDA, X1, GAMMA = best_res.x
    NSE_cal = 1 - best_res.fun
    NNSE_cal = NNSE(NSE_cal)

    # ============================================
    # Full simulation + Metrics
    # ============================================
    Qsim = GRHyMoLAP_Model([MU, LAMBDA, X1, GAMMA], Q0, Pn, En)

    NSE_val = NSE(Q_obs[b1:], Qsim[b1:])
    NNSE_val = NNSE(NSE_val)

    print(f"✅ Calibration NSE: {NSE_cal:.3f}, Validation NSE: {NSE_val:.3f}")
    print(f"   Calibration NNSE: {NNSE_cal:.3f}, Validation NNSE: {NNSE_val:.3f}")
    print(f"   Params: mu={MU:.3f}, lambda={LAMBDA:.3f}, X1={X1:.3f}, GAMMA={GAMMA:.3f}")

    # ============================================
    # Storage
    # ============================================
    results[station_id] = {
        "params": [MU, LAMBDA, X1, GAMMA],
        "NSE_cal": NSE_cal,
        "NSE_val": NSE_val,
        "NNSE_cal": NNSE_cal,
        "NNSE_val": NNSE_val,
        "Qsim": Qsim,
        "missing_ratio": missing_ratio,
        "missing_count": missing_count,
    }

print(f"\n✅ Simulation completed for {len(results)} valid basins.")



=== Station 912101A ===, Number =  1
✅ Calibration NSE: 0.529, Validation NSE: 0.574
   Calibration NNSE: 0.680, Validation NNSE: 0.701
   Params: mu=0.859, lambda=2.665, X1=112.080, GAMMA=0.112

=== Station 912105A ===, Number =  2
✅ Calibration NSE: 0.649, Validation NSE: 0.654
   Calibration NNSE: 0.740, Validation NNSE: 0.743
   Params: mu=1.045, lambda=5.433, X1=148.227, GAMMA=0.117

=== Station 915011A ===, Number =  3
✅ Calibration NSE: 0.540, Validation NSE: 0.710
   Calibration NNSE: 0.685, Validation NNSE: 0.775
   Params: mu=1.027, lambda=2.445, X1=195.836, GAMMA=0.139

=== Station 917107A ===, Number =  4
✅ Calibration NSE: 0.803, Validation NSE: 0.678
   Calibration NNSE: 0.836, Validation NNSE: 0.757
   Params: mu=1.121, lambda=4.232, X1=203.481, GAMMA=0.120

=== Station 919003A ===, Number =  5
✅ Calibration NSE: 0.621, Validation NSE: 0.722
   Calibration NNSE: 0.725, Validation NNSE: 0.782
   Params: mu=1.036, lambda=7.022, X1=128.921, GAMMA=0.147

=== Station 919201A

/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])


✅ Calibration NSE: 0.553, Validation NSE: 0.378
   Calibration NNSE: 0.691, Validation NNSE: 0.617
   Params: mu=0.961, lambda=2.387, X1=304.731, GAMMA=0.105

=== Station G9030250 ===, Number =  12


/tmp/ipython-input-2306143575.py:74: RuntimeWarning: divide by zero encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: divide by zero encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: divide by zero encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])


✅ Calibration NSE: 0.726, Validation NSE: 0.635
   Calibration NNSE: 0.785, Validation NNSE: 0.732
   Params: mu=0.969, lambda=10.962, X1=174.456, GAMMA=0.022

=== Station G9070142 ===, Number =  13
✅ Calibration NSE: 0.686, Validation NSE: 0.584
   Calibration NNSE: 0.761, Validation NNSE: 0.706
   Params: mu=1.020, lambda=4.040, X1=211.482, GAMMA=0.091

=== Station A0020101 ===, Number =  14


...

✅ Calibration NSE: 0.702, Validation NSE: 0.567
   Calibration NNSE: 0.770, Validation NNSE: 0.698
   Params: mu=0.940, lambda=6.214, X1=433.774, GAMMA=0.012

=== Station 616065 ===, Number =  197


/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: overflow encountered in scalar power
  Q[t

✅ Calibration NSE: 0.834, Validation NSE: 0.451
   Calibration NNSE: 0.858, Validation NNSE: 0.645
   Params: mu=0.877, lambda=7.931, X1=823.152, GAMMA=0.013

=== Station 803003 ===, Number =  198
✅ Calibration NSE: 0.475, Validation NSE: nan
   Calibration NNSE: 0.656, Validation NNSE: nan
   Params: mu=1.172, lambda=6.556, X1=263.186, GAMMA=0.313

=== Station 804001 ===, Number =  199
✅ Calibration NSE: 0.684, Validation NSE: 0.680
   Calibration NNSE: 0.760, Validation NNSE: 0.758
   Params: mu=1.093, lambda=5.417, X1=187.442, GAMMA=0.142

=== Station 809310 ===, Number =  200
✅ Calibration NSE: 0.453, Validation NSE: 0.659
   Calibration NNSE: 0.646, Validation NNSE: 0.745
   Params: mu=1.056, lambda=2.455, X1=138.554, GAMMA=0.254

=== Station G8110004 ===, Number =  201
✅ Calibration NSE: 0.520, Validation NSE: 0.620
   Calibration NNSE: 0.676, Validation NNSE: 0.725
   Params: mu=1.110, lambda=8.720, X1=253.462, GAMMA=0.079

=== Station G8110016 ===, Number =  202
✅ Calibration N

/tmp/ipython-input-2306143575.py:74: RuntimeWarning: divide by zero encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: divide by zero encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: divide by zero encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: divide by zero encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])
/tmp/ipython-input-2306143575.py:74: RuntimeWarning: divide by zero encountered in scalar power
  Q[t+1] = max(0, Q[t] - (MU / LAMBDA) * (Q[t])**(2*MU - 1) + gamma * Perc[t+1] * Pn[t+1])


✅ Calibration NSE: 0.838, Validation NSE: 0.813
   Calibration NNSE: 0.860, Validation NNSE: 0.843
   Params: mu=0.891, lambda=10.800, X1=177.338, GAMMA=0.026

=== Station G8140161 ===, Number =  205
✅ Calibration NSE: 0.638, Validation NSE: 0.698
   Calibration NNSE: 0.734, Validation NNSE: 0.768
   Params: mu=1.182, lambda=8.849, X1=426.676, GAMMA=0.051

=== Station G8150018 ===, Number =  206
✅ Calibration NSE: 0.834, Validation NSE: 0.863
   Calibration NNSE: 0.858, Validation NNSE: 0.880
   Params: mu=1.124, lambda=6.102, X1=404.768, GAMMA=0.149

=== Station G8170002 ===, Number =  207
✅ Calibration NSE: 0.675, Validation NSE: 0.761
   Calibration NNSE: 0.755, Validation NNSE: 0.807
   Params: mu=0.976, lambda=1.926, X1=277.609, GAMMA=0.170

=== Station G8190001 ===, Number =  208
✅ Calibration NSE: 0.579, Validation NSE: 0.648
   Calibration NNSE: 0.704, Validation NNSE: 0.740
   Params: mu=1.038, lambda=3.524, X1=329.047, GAMMA=0.180

=== Station G8200045 ===, Number =  209
✅ Ca

NSE result summary

In [ ]:
# =============================================================
# 📌 EXTRACTION DES NSE & NNSE — CALIBRATION
# =============================================================
nse_cal = [res['NSE_cal'] for res in results.values()]
nnse_cal = [res['NNSE_cal'] for res in results.values()]

print("\n================= CALIBRATION =================\n")

if nse_cal:
    print(f"NSE Calibration -> Median: {np.percentile(nse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_cal, 95):.4f}")
    print("MEAN NSE_CAL:", np.mean(nse_cal))
    print("MIN NSE_CAL:", np.min(nse_cal))
    print("MAX NSE_CAL:", np.max(nse_cal))
else:
    print("No NSE available for the calibration.")

print("\n--- NNSE Calibration ---")
if nnse_cal:
    print(f"NNSE Calibration -> Median: {np.percentile(nnse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(nnse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(nnse_cal, 95):.4f}")
    print("MEAN NNSE_CAL:", np.mean(nnse_cal))
    print("MIN NNSE_CAL:", np.min(nnse_cal))
    print("MAX NNSE_CAL:", np.max(nnse_cal))
else:
    print("No NNSE avalaible for the calibration.")


# =============================================================
# 📌 EXTRACTION DES NSE & NNSE — VALIDATION
# =============================================================
print("\n\n================= VALIDATION =================\n")

nse_val = [res['NSE_val'] for res in results.values() if not np.isnan(res['NSE_val'])]
nnse_val = [res['NNSE_val'] for res in results.values() if not np.isnan(res['NNSE_val'])]

if nse_val:
    print(f"NSE Validation -> Median: {np.percentile(nse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_val, 95):.4f}")
    print("MEAN NSE_VAL:", np.mean(nse_val))
    print("MIN NSE_VAL:", np.min(nse_val))
    print("MAX NSE_VAL:", np.max(nse_val))
else:
    print("No valid station for NSE in validation.")

print("\n--- NNSE Validation ---")
if nnse_val:
    print(f"NNSE Validation -> Median: {np.percentile(nnse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(nnse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(nnse_val, 95):.4f}")
    print("MEAN NNSE_VAL:", np.mean(nnse_val))
    print("MIN NNSE_VAL:", np.min(nnse_val))
    print("MAX NNSE_VAL:", np.max(nnse_val))
else:
    print("No valid station for NNSE in validation.")


================= CALIBRATION =================

NSE Calibration -> Median: 0.707, 5th percentile: 0.4707, 95th percentile: 0.8579
MEAN NSE_CAL: 0.6955610490171391
MIN NSE_CAL: -0.06045121141769827
MAX NSE_CAL: 0.8965007287694883

--- NNSE Calibration ---
NNSE Calibration -> Median: 0.773, 5th percentile: 0.6539, 95th percentile: 0.8756
MEAN NNSE_CAL: 0.7733687260678712
MIN NNSE_CAL: 0.485330588979075
MAX NNSE_CAL: 0.9062081199971255


================= VALIDATION =================

NSE Validation -> Median: 0.643, 5th percentile: 0.3148, 95th percentile: 0.8424
MEAN NSE_VAL: 0.5995153859717214
MIN NSE_VAL: -2.75323433972195
MAX NSE_VAL: 0.9022049314123964

--- NNSE Validation ---
NNSE Validation -> Median: 0.737, 5th percentile: 0.5934, 95th percentile: 0.8638
MEAN NNSE_VAL: 0.7312481954382631
MIN NNSE_VAL: 0.2103830630952012
MAX NNSE_VAL: 0.9109168264770724
